# Phase 3 — Nepali Reasoning Fine-tuning (Self-Contained Kaggle Notebook)

**Language:** Nepali | **Model:** DecoderOnlyLM (Phase 2, 25.8M params)

All source code is inlined. No GitHub clone required.

**Inputs you must attach to this notebook:**
- `notebooks/negative00/notebookb7b29f6528` (Phase 2 Nepali checkpoint)
- `datasets/negative00/phase2-cloud-data` (Nepali tokenizer)
- `datasets/negative00/phase-3-finetune-dataset` (Nepali JSONL train/val/test)

> ⛔ `RUN_FINETUNING` defaults to `False`. Baseline is safe to run immediately.

## Section 1 — Configuration

In [ ]:
# ================================================================
# CONFIGURATION — edit this cell only
# ================================================================
LANGUAGE = "nepali"

CHECKPOINT_PATH     = "/kaggle/input/notebooks/negative00/notebookb7b29f6528/phase2_outputs/nepali_checkpoints/step_00020000.pt"
TOKENIZER_PATH      = "/kaggle/input/datasets/negative00/phase2-cloud-data/nepali/tokenizer/models/nepali_bpe_10k_assignment.model"
TRAIN_PATH          = "/kaggle/input/datasets/negative00/phase-3-finetune-dataset/nepali_train.jsonl"
VAL_PATH            = "/kaggle/input/datasets/negative00/phase-3-finetune-dataset/nepali_val.jsonl"
TEST_PATH           = "/kaggle/input/datasets/negative00/phase-3-finetune-dataset/nepali_test.jsonl"
OUTPUT_DIR          = "/kaggle/working/results/phase3_finetune/nepali/"
BASELINE_JSON       = OUTPUT_DIR + "baseline_nepali.json"
FINETUNED_EVAL_JSON = OUTPUT_DIR + "finetuned_nepali.json"

EXPECTED_TRAIN = 9600
EXPECTED_VAL   = 1200
EXPECTED_TEST  = 1200

# ---- Fine-tuning hyperparameters ----
FT_MAX_STEPS         = 3000
FT_WARMUP_STEPS      = 300
FT_LEARNING_RATE     = 5e-5
FT_MIN_LR            = 5e-6
FT_BATCH_SIZE        = 16
FT_GRAD_ACCUM_STEPS  = 1
FT_WEIGHT_DECAY      = 0.1
FT_GRAD_CLIP         = 1.0
FT_EVAL_INTERVAL     = 500
FT_EVAL_BATCHES      = 50
FT_CKPT_INTERVAL     = 1000
FT_LOG_INTERVAL      = 10
SEED                 = 42

# ---- Safety flags ----
RUN_BASELINE   = True
RUN_FINETUNING = False  # Change to True ONLY when ready to train

import os
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"Language          : {LANGUAGE}")
print(f"Checkpoint        : {CHECKPOINT_PATH}")
print(f"Tokenizer         : {TOKENIZER_PATH}")
print(f"Train             : {TRAIN_PATH}")
print(f"Val               : {VAL_PATH}")
print(f"Test              : {TEST_PATH}")
print(f"Output dir        : {OUTPUT_DIR}")
print(f"RUN_BASELINE      : {RUN_BASELINE}")
print(f"RUN_FINETUNING    : {RUN_FINETUNING}")

## Section 2 — Environment Setup

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "sentencepiece", "-q"], check=True)

import torch
print(f"PyTorch  : {torch.__version__}")
print(f"CUDA     : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU      : {torch.cuda.get_device_name(0)}")
    print(f"VRAM     : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device   : {DEVICE}")

## Section 3 — Inline Model Definition (Phase 2 Architecture)

In [ ]:
import math
import torch
import torch.nn as nn
from dataclasses import dataclass

@dataclass
class LMConfig:
    vocab_size: int = 10000
    d_model: int = 448
    n_layers: int = 8
    n_heads: int = 7
    d_ff: int = 2048
    max_seq_len: int = 512
    dropout: float = 0.1
    bias: bool = True

class InputEmbeddings(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.max_seq_len = config.max_seq_len
        self.token_embedding    = nn.Embedding(config.vocab_size, config.d_model)
        self.position_embedding = nn.Embedding(config.max_seq_len, config.d_model)
        self.dropout = nn.Dropout(config.dropout)
    def forward(self, input_ids):
        B, T = input_ids.shape
        pos = torch.arange(T, dtype=torch.long, device=input_ids.device)
        return self.dropout(self.token_embedding(input_ids) + self.position_embedding(pos))

class CausalSelfAttention(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.n_heads  = config.n_heads
        self.head_dim = config.d_model // config.n_heads
        self.max_seq_len = config.max_seq_len
        self.q_proj   = nn.Linear(config.d_model, config.d_model, bias=config.bias)
        self.k_proj   = nn.Linear(config.d_model, config.d_model, bias=config.bias)
        self.v_proj   = nn.Linear(config.d_model, config.d_model, bias=config.bias)
        self.out_proj = nn.Linear(config.d_model, config.d_model, bias=config.bias)
        self.attn_dropout  = nn.Dropout(config.dropout)
        self.resid_dropout = nn.Dropout(config.dropout)
        mask = torch.tril(torch.ones(config.max_seq_len, config.max_seq_len)).view(1,1,config.max_seq_len,config.max_seq_len)
        self.register_buffer("bias_mask", mask)
    def forward(self, x, return_attention=False):
        B, T, C = x.size()
        q = self.q_proj(x).view(B,T,self.n_heads,self.head_dim).transpose(1,2)
        k = self.k_proj(x).view(B,T,self.n_heads,self.head_dim).transpose(1,2)
        v = self.v_proj(x).view(B,T,self.n_heads,self.head_dim).transpose(1,2)
        att = (q @ k.transpose(-2,-1)) * (1.0/math.sqrt(self.head_dim))
        att = att.masked_fill(self.bias_mask[:,:,:T,:T]==0, float('-inf'))
        att = torch.nn.functional.softmax(att, dim=-1)
        y   = self.attn_dropout(att) @ v
        y   = y.transpose(1,2).contiguous().view(B,T,C)
        y   = self.resid_dropout(self.out_proj(y))
        if return_attention: return y, att
        return y

class FeedForward(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(config.d_model, config.d_ff,    bias=config.bias), nn.GELU(), nn.Dropout(config.dropout),
            nn.Linear(config.d_ff,    config.d_model, bias=config.bias), nn.Dropout(config.dropout))
    def forward(self, x): return self.net(x)

class TransformerBlock(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.ln_1 = nn.LayerNorm(config.d_model)
        self.attn = CausalSelfAttention(config)
        self.ln_2 = nn.LayerNorm(config.d_model)
        self.ffn  = FeedForward(config)
    def forward(self, x, return_attention=False):
        if return_attention:
            a, w = self.attn(self.ln_1(x), return_attention=True)
            return x + a + self.ffn(self.ln_2(x + a)), w
        return x + self.attn(self.ln_1(x)) + self.ffn(self.ln_2(x + self.attn(self.ln_1(x))))

class DecoderOnlyLM(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config  = config
        self.embedding = InputEmbeddings(config)
        self.blocks    = nn.ModuleList([TransformerBlock(config) for _ in range(config.n_layers)])
        self.ln_f      = nn.LayerNorm(config.d_model)
        self.lm_head   = nn.Linear(config.d_model, config.vocab_size, bias=config.bias)
        self.lm_head.weight = self.embedding.token_embedding.weight
    def forward(self, input_ids, return_attention=False):
        x = self.embedding(input_ids)
        for block in self.blocks:
            x = block(x)
        return self.lm_head(self.ln_f(x))

print("Model definition loaded.")
cfg = LMConfig()
_tmp = DecoderOnlyLM(cfg)
print(f"Parameters: {sum(p.numel() for p in _tmp.parameters()):,}")
del _tmp

## Section 4 — Inline Training Infrastructure

In [ ]:
import json, math, os, random, shutil, tempfile, time
from pathlib import Path
from typing import Optional
import numpy as np
import torch, torch.nn as nn
from torch.utils.data import DataLoader, Dataset

CHECKPOINT_VERSION = 1

def seed_everything(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)

def build_optimizer(model, cfg):
    decay, no_decay = [], []
    for n, p in model.named_parameters():
        if not p.requires_grad: continue
        (no_decay if p.ndim <= 1 else decay).append(p)
    return torch.optim.AdamW(
        [{"params": decay, "weight_decay": cfg['weight_decay']},
         {"params": no_decay, "weight_decay": 0.0}],
        lr=cfg['learning_rate'], betas=(cfg['beta1'], cfg['beta2']), eps=cfg['eps'])

def get_lr(step, cfg):
    lr, min_lr, ws, ms = cfg['learning_rate'], cfg['min_lr'], cfg['warmup_steps'], cfg['max_steps']
    if step < ws: return lr * step / max(1, ws)
    if step >= ms: return min_lr
    p = (step - ws) / max(1, ms - ws)
    return min_lr + 0.5*(1+math.cos(math.pi*p))*(lr - min_lr)

def build_scheduler(optimizer, cfg):
    return torch.optim.lr_scheduler.LambdaLR(
        optimizer, lambda s: get_lr(s, cfg)/cfg['learning_rate'])

def compute_loss(logits, targets):
    B, T, V = logits.shape
    return nn.functional.cross_entropy(logits.view(B*T,V), targets.view(B*T).long(), ignore_index=-100)

def perplexity(loss):
    try: return math.exp(min(loss, 20.0))
    except: return float('inf')

def get_rng_state():
    s = {'python_rng': random.getstate(), 'numpy_rng': np.random.get_state(), 'torch_rng': torch.get_rng_state()}
    if torch.cuda.is_available(): s['cuda_rng'] = torch.cuda.get_rng_state_all()
    return s

def set_rng_state(s):
    random.setstate(s['python_rng']); np.random.set_state(s['numpy_rng']); torch.set_rng_state(s['torch_rng'])
    if torch.cuda.is_available() and 'cuda_rng' in s: torch.cuda.set_rng_state_all(s['cuda_rng'])

def save_checkpoint(output_dir, step, model, optimizer, scheduler, cfg, model_cfg, best_val_loss=None, rng_state=None):
    out = Path(output_dir); out.mkdir(parents=True, exist_ok=True)
    payload = {'version': CHECKPOINT_VERSION, 'step': step,
                'model_state': model.state_dict(), 'optimizer_state': optimizer.state_dict(),
                'scheduler_state': scheduler.state_dict(), 'train_cfg': cfg,
                'model_cfg': vars(model_cfg), 'best_val_loss': best_val_loss, 'rng_state': rng_state}
    name = f'step_{step:08d}.pt'; final = out/name
    fd, tmp = tempfile.mkstemp(dir=out, suffix='.tmp')
    try:
        os.close(fd); torch.save(payload, tmp); shutil.move(tmp, final)
    except:
        if os.path.exists(tmp): os.remove(tmp); raise
    (out/'latest.txt').write_text(name)
    return final

def load_checkpoint(ckpt_path, model, optimizer=None, scheduler=None, device=None, restore_rng=True):
    if device is None: device = torch.device('cpu')
    payload = torch.load(ckpt_path, map_location=device, weights_only=False)
    if payload.get('version') != CHECKPOINT_VERSION:
        raise ValueError(f"Checkpoint version mismatch: expected {CHECKPOINT_VERSION}, got {payload.get('version')}")
    model.load_state_dict(payload['model_state'])
    if optimizer:  optimizer.load_state_dict(payload['optimizer_state'])
    if scheduler:  scheduler.load_state_dict(payload['scheduler_state'])
    if restore_rng and payload.get('rng_state'): set_rng_state(payload['rng_state'])
    return payload

def _amp_ok(cfg, device): return cfg.get('use_amp', False) and device.type == 'cuda'

class InfiniteLoader:
    def __init__(self, ds, bs): self.ds=ds; self.bs=bs; self._reset()
    def _reset(self): self._iter=iter(DataLoader(self.ds, batch_size=self.bs, shuffle=True, drop_last=True, num_workers=0))
    def __next__(self):
        try: return next(self._iter)
        except StopIteration: self._reset(); return next(self._iter)

@torch.no_grad()
def evaluate_model(model, dataset, cfg, device, use_amp):
    model.eval()
    loader = DataLoader(dataset, batch_size=cfg['batch_size'], shuffle=True, drop_last=True, num_workers=0)
    total, n = 0.0, 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
            total += compute_loss(model(x), y).item()
        n += 1
        if n >= cfg['eval_batches']: break
    model.train()
    m = total/max(n,1)
    return {'val_loss': m, 'val_ppl': perplexity(m)}

print("Training infrastructure loaded.")

## Section 5 — Inline Dataset Loader (Answer-Only Masking)

In [ ]:
import json, torch, sentencepiece as spm
from torch.utils.data import Dataset

class Phase3ReasoningDataset(Dataset):
    """JSONL loader with answer-only loss masking (prompt tokens -> -100)."""
    def __init__(self, jsonl_path, tokenizer_path, max_length=512):
        self.max_length = max_length
        self.sp = spm.SentencePieceProcessor()
        self.sp.Load(str(tokenizer_path))
        self.examples = []
        with open(jsonl_path, encoding='utf-8') as f:
            for line in f:
                if line.strip(): self.examples.append(json.loads(line))

    def __len__(self): return len(self.examples)

    def __getitem__(self, idx):
        d = self.examples[idx]
        prompt = f"\u092a\u094d\u0930\u0936\u094d\u0928: {d['premise']} {d['question']} \u0909\u0924\u094d\u0924\u0930: "
        full   = prompt + str(d['answer'])
        prompt_ids = self.sp.Encode(prompt, out_type=int)
        full_ids   = self.sp.Encode(full,   out_type=int) + [self.sp.eos_id()]
        if len(full_ids) > self.max_length+1:
            full_ids = full_ids[:self.max_length+1]
        x = full_ids[:-1]; y = list(full_ids[1:])
        L = len(prompt_ids)
        for i in range(L-1):
            if i < len(y): y[i] = -100
        pad = self.max_length - len(x)
        if pad > 0:
            x = x + [0]*pad; y = y + [-100]*pad
        return torch.tensor(x, dtype=torch.long), torch.tensor(y, dtype=torch.long)

print("Dataset class loaded.")

## Section 6 — Tokenizer Verification

In [ ]:
import os, sentencepiece as spm
if not os.path.exists(TOKENIZER_PATH):
    raise FileNotFoundError(f"Nepali tokenizer not found:\n  {TOKENIZER_PATH}")
sp = spm.SentencePieceProcessor()
sp.Load(TOKENIZER_PATH)
assert sp.GetPieceSize() == 10000, f"Expected vocab 10000, got {sp.GetPieceSize()}"
sample = "\u0930\u093e\u092e \u0936\u094d\u092f\u093e\u092e \u092d\u0928\u094d\u0926\u093e \u0905\u0917\u094d\u0932\u094b \u091b \u0964"
print(f"Vocab size   : {sp.GetPieceSize()} (expected 10000) \u2713")
print(f"EOS id       : {sp.eos_id()}")
print(f"Smoke decode : {sp.Decode(sp.Encode(sample))}")
print("Tokenizer verified.")

## Section 7 — Dataset Verification

In [ ]:
import json, os
REQUIRED = {'task_type','attribute','question_category','template_id','premise','question','answer','id','language','split'}

def verify_split(path, expected, name):
    if not os.path.exists(path): raise FileNotFoundError(f"{name} not found: {path}")
    rows=[json.loads(l) for l in open(path, encoding='utf-8') if l.strip()]
    assert len(rows)==expected, f"{name}: expected {expected}, got {len(rows)}"
    lens, max_unk = [], 0
    for d in rows:
        assert not (REQUIRED-d.keys()), f"Missing fields in {d.get('id','?')}"
        assert d['language']==LANGUAGE, f"Language mismatch: {d['language']}"
        text = f"\u092a\u094d\u0930\u0936\u094d\u0928: {d['premise']} {d['question']} \u0909\u0924\u094d\u0924\u0930: {d['answer']}"
        ids = sp.Encode(text, out_type=int)+[sp.eos_id()]
        max_unk=max(max_unk, ids.count(sp.unk_id())); lens.append(len(ids))
    print(f"  {name}: {len(rows)} \u2713 | max_tok={max(lens)} | mean={sum(lens)/len(lens):.1f} | over_512={sum(1 for l in lens if l>512)} | max_unk={max_unk}")

verify_split(TRAIN_PATH, EXPECTED_TRAIN, 'TRAIN')
verify_split(VAL_PATH,   EXPECTED_VAL,   'VAL')
verify_split(TEST_PATH,  EXPECTED_TEST,  'TEST')
print("Dataset verification passed.")

## Section 8 — Checkpoint Verification

In [ ]:
import os, torch
if not os.path.exists(CHECKPOINT_PATH):
    raise FileNotFoundError(
        f"Phase 2 Nepali checkpoint not found:\n  {CHECKPOINT_PATH}\n"
        "Attach the notebook output containing the checkpoint before running.")

print(f"Loading: {CHECKPOINT_PATH}")
payload = torch.load(CHECKPOINT_PATH, map_location='cpu', weights_only=False)
assert payload.get('version') == CHECKPOINT_VERSION, f"Version mismatch: {payload.get('version')}"
ckpt_step = payload.get('step', 'N/A')
print(f"Step    : {ckpt_step}")
if ckpt_step != 'N/A': assert ckpt_step == 20000, f"Expected step 20000, got {ckpt_step}"

saved_cfg = payload.get('model_cfg', {})
for k,v in dict(vocab_size=10000,d_model=448,n_layers=8,n_heads=7,d_ff=2048,max_seq_len=512).items():
    a = saved_cfg.get(k)
    print(f"  {k:12s}: {v} {'\u2713' if a==v else f'\u2717 got {a}'}")

ref = DecoderOnlyLM(LMConfig()); ref_keys=set(ref.state_dict().keys()); ckpt_keys=set(payload['model_state'].keys())
missing=ref_keys-ckpt_keys; extra=ckpt_keys-ref_keys
print(f"Missing keys   : {missing or 'none \u2713'}")
print(f"Unexpected keys: {extra or 'none \u2713'}")
del ref
print("Checkpoint verified.")

## Section 9 — Pretrained Baseline Evaluation

In [ ]:
import json, os, torch, re
from collections import defaultdict

_UNK_SURFACE = '⁇'

def extract_answer(raw_continuation):
    text = raw_continuation.lstrip()
    match = re.match(r"^([^\s.,;!?\u0964]+)", text)
    if not match:
        return ''
    extracted = match.group(1)
    if extracted == _UNK_SURFACE:
        return ''
    return extracted

def generate_answer(model, sp, prompt_text, max_new=20, device='cpu'):
    model.eval()
    ids = torch.tensor([sp.Encode(prompt_text, out_type=int)], dtype=torch.long, device=device)
    gen = []
    with torch.no_grad():
        for _ in range(max_new):
            if ids.shape[1] > 512: ids = ids[:, -512:]
            nxt = model(ids)[0,-1,:].argmax().item()
            if nxt == sp.eos_id(): break
            gen.append(nxt)
            ids = torch.cat([ids, torch.tensor([[nxt]], device=device)], dim=1)
    return sp.Decode(gen)

def run_eval(ckpt_path, test_path, lang, sp, device):
    model = DecoderOnlyLM(LMConfig()).to(device)
    load_checkpoint(ckpt_path, model, device=device, restore_rng=False)
    rows = [json.loads(l) for l in open(test_path, encoding='utf-8') if l.strip()]
    correct=0; total=0
    by_task=defaultdict(lambda:[0,0]); by_attr=defaultdict(lambda:[0,0])
    by_cat=defaultdict(lambda:[0,0]);  by_heldout=defaultdict(lambda:[0,0])
    by_tmpl=defaultdict(lambda:[0,0])
    for i,d in enumerate(rows):
        prompt = f"प्रश्न: {d['premise']} {d['question']} उत्तर: "
        raw_pred = generate_answer(model, sp, prompt, device=device)
        pred = extract_answer(raw_pred)
        gt   = str(d['answer']).strip()
        ok   = int(pred == gt)
        total+=1; correct+=ok
        task=d.get('task_type','?'); attr=d.get('attribute','?')
        cat=d.get('question_category','?'); tmpl=d.get('template_id','?')
        hk='held_out' if d.get('is_held_out_template') else 'familiar'
        for store, key in [(by_task,task),(by_attr,attr),(by_cat,cat),(by_tmpl,tmpl),(by_heldout,hk)]:
            store[key][1]+=1; store[key][0]+=ok
        if (i+1)%100==0: print(f"  {i+1}/{len(rows)} acc={correct/total*100:.1f}%")
    results = {
        'overall_accuracy': correct/total if total > 0 else 0, 'correct': correct, 'total': total,
        'by_task':    {k:v[0]/v[1] for k,v in by_task.items()},
        'by_attr':    {k:v[0]/v[1] for k,v in by_attr.items()},
        'by_category':{k:v[0]/v[1] for k,v in by_cat.items()},
        'by_template':{k:v[0]/v[1] for k,v in by_tmpl.items()},
        'by_held_out':{k:v[0]/v[1] for k,v in by_heldout.items()},
    }
    return results


if not RUN_BASELINE:
    print("RUN_BASELINE is False — skipping.")
else:
    device = torch.device(DEVICE)
    print(f"Running baseline on {EXPECTED_TEST} test examples...")
    baseline_results = run_eval(CHECKPOINT_PATH, TEST_PATH, LANGUAGE, sp, device)
    with open(BASELINE_JSON, 'w', encoding='utf-8') as f:
        json.dump(baseline_results, f, indent=2, ensure_ascii=False)
    print(f"\nBaseline saved to {BASELINE_JSON}")
    print(f"Overall accuracy: {baseline_results['overall_accuracy']*100:.2f}%")


## Section 10 — Display Baseline Results

In [ ]:
import json, os
if not RUN_BASELINE or not os.path.exists(BASELINE_JSON):
    print("No baseline results available.")
else:
    r = json.load(open(BASELINE_JSON, encoding='utf-8'))
    print("="*55)
    print("  NEPALI PRETRAINED BASELINE — EXACT MATCH")
    print("="*55)
    print(f"  Overall : {r['overall_accuracy']*100:.2f}%  ({r['correct']}/{r['total']})")
    for gk, gl in [('by_task','By Task'),('by_attr','By Attribute'),
                   ('by_category','By Category'),('by_held_out','Familiar vs Held-Out')]:
        if gk in r:
            print(f"\n  {gl}:")
            for k,v in sorted(r[gk].items()): print(f"    {k:35s}: {v*100:.2f}%")

## Section 11 — Fine-tuning
> ⛔ Set `RUN_FINETUNING = True` in Section 1 to enable.

In [ ]:
import os, torch

if not RUN_FINETUNING:
    print("\u26d4 RUN_FINETUNING is False. Fine-tuning will NOT run.")
    print("   Set RUN_FINETUNING = True in Section 1 to enable.")
else:
    if not os.path.exists(CHECKPOINT_PATH):
        raise FileNotFoundError(f"Checkpoint required for fine-tuning: {CHECKPOINT_PATH}")

    ft_cfg = {
        'language': LANGUAGE, 'seed': SEED,
        'learning_rate': FT_LEARNING_RATE, 'min_lr': FT_MIN_LR,
        'beta1': 0.9, 'beta2': 0.95, 'eps': 1e-8,
        'weight_decay': FT_WEIGHT_DECAY, 'grad_clip': FT_GRAD_CLIP,
        'batch_size': FT_BATCH_SIZE, 'grad_accum_steps': FT_GRAD_ACCUM_STEPS,
        'max_steps': FT_MAX_STEPS, 'warmup_steps': FT_WARMUP_STEPS,
        'eval_interval': FT_EVAL_INTERVAL, 'eval_batches': FT_EVAL_BATCHES,
        'checkpoint_interval': FT_CKPT_INTERVAL, 'log_interval': FT_LOG_INTERVAL,
        'use_amp': True,
    }

    device = torch.device(DEVICE)
    seed_everything(SEED)

    print("Loading pretrained checkpoint...")
    model_cfg = LMConfig()
    model = DecoderOnlyLM(model_cfg).to(device)
    payload = load_checkpoint(CHECKPOINT_PATH, model, device=device, restore_rng=False)
    print(f"Loaded from step {payload.get('step', '?')}")

    print("Loading datasets...")
    train_ds = Phase3ReasoningDataset(TRAIN_PATH, TOKENIZER_PATH)
    val_ds   = Phase3ReasoningDataset(VAL_PATH,   TOKENIZER_PATH)
    print(f"Train: {len(train_ds)} | Val: {len(val_ds)}")

    optimizer = build_optimizer(model, ft_cfg)
    scheduler = build_scheduler(optimizer, ft_cfg)
    use_amp   = _amp_ok(ft_cfg, device)
    scaler    = torch.amp.GradScaler('cuda', enabled=use_amp)
    batch_iter = InfiniteLoader(train_ds, FT_BATCH_SIZE)
    best_val   = float('inf')

    print(f"\nStarting fine-tuning | steps={FT_MAX_STEPS} | lr={FT_LEARNING_RATE} | amp={use_amp}")
    for step in range(FT_MAX_STEPS):
        model.train()
        optimizer.zero_grad(set_to_none=True)
        x, y = next(batch_iter)
        x, y = x.to(device), y.to(device)
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
            loss = compute_loss(model(x), y)
        if use_amp: scaler.scale(loss).backward()
        else:       loss.backward()
        if FT_GRAD_CLIP > 0:
            if use_amp: scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), FT_GRAD_CLIP)
        if use_amp: scaler.step(optimizer); scaler.update()
        else: optimizer.step()
        scheduler.step()
        cur = step+1
        if cur % FT_LOG_INTERVAL == 0:
            print(f"step {cur:5d} | loss {loss.item()*FT_GRAD_ACCUM_STEPS:.4f} | lr {scheduler.get_last_lr()[0]:.2e}")
        if cur % FT_EVAL_INTERVAL == 0:
            em = evaluate_model(model, val_ds, ft_cfg, device, use_amp)
            print(f"  [eval] val_loss={em['val_loss']:.4f} | ppl={em['val_ppl']:.2f}")
            if em['val_loss'] < best_val: best_val = em['val_loss']
        if cur % FT_CKPT_INTERVAL == 0:
            ckpt = save_checkpoint(OUTPUT_DIR, cur, model, optimizer, scheduler, ft_cfg, model_cfg, best_val, get_rng_state())
            print(f"  [ckpt] {ckpt}")

    print("\nFine-tuning complete.")

## Section 12 — Post-Fine-tuning Evaluation

In [ ]:
import json, os, torch

if not RUN_FINETUNING:
    print("RUN_FINETUNING is False — skipping post-fine-tuning evaluation.")
else:
    latest_txt = os.path.join(OUTPUT_DIR, 'latest.txt')
    if not os.path.exists(latest_txt):
        raise FileNotFoundError(f"latest.txt not found in {OUTPUT_DIR}")
    ft_ckpt = os.path.join(OUTPUT_DIR, open(latest_txt).read().strip())
    print(f"Evaluating fine-tuned: {ft_ckpt}")
    device = torch.device(DEVICE)
    ft_results = run_eval(ft_ckpt, TEST_PATH, LANGUAGE, sp, device)
    with open(FINETUNED_EVAL_JSON, 'w', encoding='utf-8') as f:
        json.dump(ft_results, f, indent=2, ensure_ascii=False)
    print(f"Fine-tuned results saved to {FINETUNED_EVAL_JSON}")

## Section 13 — Compare Pretrained vs Fine-tuned

In [ ]:
import json, os
def load_j(p): return json.load(open(p,encoding='utf-8')) if os.path.exists(p) else None
pre = load_j(BASELINE_JSON); ft = load_j(FINETUNED_EVAL_JSON)
print("="*55)
print("  NEPALI — PRETRAINED vs FINE-TUNED")
print("="*55)
for label, r in [("Pretrained", pre),("Fine-tuned", ft)]:
    if r: print(f"  {label:12s}: {r['overall_accuracy']*100:.2f}%  ({r['correct']}/{r['total']})")
    else: print(f"  {label:12s}: NOT AVAILABLE")
if pre and ft:
    d = (ft['overall_accuracy']-pre['overall_accuracy'])*100
    print(f"\n  Delta: {d:+.2f}% ({'improved' if d>0 else 'decreased'})")

## Section 14 — Save Run Summary

In [ ]:
import json, datetime, os
summary = {
    'language': LANGUAGE, 'timestamp': datetime.datetime.utcnow().isoformat()+'Z',
    'checkpoint': CHECKPOINT_PATH, 'tokenizer': TOKENIZER_PATH, 'test': TEST_PATH,
    'baseline_json': BASELINE_JSON, 'finetuned_json': FINETUNED_EVAL_JSON,
    'run_baseline': RUN_BASELINE, 'run_finetuning': RUN_FINETUNING, 'seed': SEED,
}
p = os.path.join(OUTPUT_DIR, 'run_summary_nepali.json')
json.dump(summary, open(p,'w',encoding='utf-8'), indent=2, ensure_ascii=False)
print(f"Summary saved: {p}")
print(json.dumps(summary, indent=2, ensure_ascii=False))